<a href="https://colab.research.google.com/github/Raksh1707/taskdeeplearning/blob/main/task11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import math


In [2]:
img_size = 32
patch_size = 8
channels = 3
embed_dim = 64
heads = 4

# Image
x = torch.randn(2, 3, 32, 32)

In [3]:
patches = x.unfold(2, patch_size, patch_size)
patches = patches.unfold(3, patch_size, patch_size)


In [4]:
patches = patches.permute(0, 2, 3, 1, 4, 5)

# Flatten patches
patches = patches.reshape(2, -1, channels * patch_size * patch_size)

print("Patch shape:", patches.shape)

Patch shape: torch.Size([2, 16, 192])


In [5]:
projection = nn.Linear(
    channels * patch_size * patch_size,
    embed_dim
)

tokens = projection(patches)

print("Projected shape:", tokens.shape)


Projected shape: torch.Size([2, 16, 64])


In [6]:
cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

cls = cls_token.expand(2, -1, -1)

tokens = torch.cat((cls, tokens), dim=1)

print("After class token:", tokens.shape)

After class token: torch.Size([2, 17, 64])


In [7]:
pos_embedding = nn.Parameter(
    torch.randn(1, tokens.size(1), embed_dim)
)

tokens = tokens + pos_embedding


In [8]:
qkv = nn.Linear(embed_dim, embed_dim * 3)

q, k, v = qkv(tokens).chunk(3, dim=-1)

# Split into heads
q = q.reshape(2, tokens.size(1), heads, -1).transpose(1, 2)
k = k.reshape(2, tokens.size(1), heads, -1).transpose(1, 2)
v = v.reshape(2, tokens.size(1), heads, -1).transpose(1, 2)


In [9]:
attention = torch.matmul(q, k.transpose(-2, -1))
attention = attention / math.sqrt(q.size(-1))

attention = torch.softmax(attention, dim=-1)

output = torch.matmul(attention, v)


In [10]:
output = output.transpose(1, 2).reshape(
    2, tokens.size(1), embed_dim
)

print("Attention output:", output.shape)
print("Attention map:", attention.shape)

print("ViT completed successfully!")

Attention output: torch.Size([2, 17, 64])
Attention map: torch.Size([2, 4, 17, 17])
ViT completed successfully!
